In [2]:
from pathlib import Path
import json
import re
import html


# ----------------------------
# Find Learning Lab repo root
# ----------------------------

def find_repo_root(start_path):
    start_path = Path(start_path).resolve()

    for path in [start_path] + list(start_path.parents):
        if (path / "mkdocs.yml").exists():
            return path

    raise FileNotFoundError("Could not find mkdocs.yml. Run this from inside the Learning Lab repo.")


ROOT = find_repo_root(Path.cwd())

DATASET_REPO = ROOT.parent / "cloud-datasets"
DATASET_INDEX_FILE = DATASET_REPO / "datasets.json"
DATASET_DIR = DATASET_REPO / "datasets"

if not DATASET_INDEX_FILE.exists():
    raise FileNotFoundError(f"Could not find dataset index file: {DATASET_INDEX_FILE}")

if not DATASET_DIR.exists():
    raise FileNotFoundError(f"Could not find dataset detail folder: {DATASET_DIR}")

OUT_FILE = ROOT / "docs" / "rosetta-stone" / "datasets.md"
JS_FILE = ROOT / "docs" / "javascripts" / "dataset-filter.js"

print("Learning Lab root:", ROOT)
print("Dataset repo:", DATASET_REPO)
print("Dataset index file:", DATASET_INDEX_FILE)
print("Dataset detail folder:", DATASET_DIR)
print("Output Markdown:", OUT_FILE)
print("Output JavaScript:", JS_FILE)


# ----------------------------
# Helpers
# ----------------------------

def esc(value):
    """Escape text for safe HTML display."""
    if value is None:
        return ""
    return html.escape(str(value), quote=True)


def safe_id(value):
    """Create a safe HTML id."""
    value = str(value).lower().strip()
    value = re.sub(r"[^a-z0-9]+", "-", value)
    return value.strip("-") or "dataset"


def doi_url(doi):
    """Convert DOI text into a DOI URL."""
    if not doi:
        return ""

    doi = str(doi).strip()

    if doi.startswith("http://") or doi.startswith("https://"):
        return doi

    return f"https://doi.org/{doi}"


def doi_link(doi):
    """Create a clickable DOI link."""
    if not doi:
        return "TBD"

    doi = str(doi).strip()
    url = doi_url(doi)
    label = doi.replace("https://doi.org/", "").replace("http://doi.org/", "")

    return f'<a href="{esc(url)}" target="_blank" rel="noopener">{esc(label)}</a>'


def version_key(version):
    """Sort versions like v4.1.0 above v4.0.0."""
    numbers = re.findall(r"\d+", str(version))

    if not numbers:
        return (0,)

    return tuple(int(number) for number in numbers)


def load_json_file(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def deep_merge(base, extra):
    """
    Merge two dictionaries.
    Values from extra override base.
    Nested dictionaries are merged.
    """
    merged = dict(base)

    for key, value in extra.items():
        if (
            key in merged
            and isinstance(merged[key], dict)
            and isinstance(value, dict)
        ):
            merged[key] = deep_merge(merged[key], value)
        else:
            merged[key] = value

    return merged


# ----------------------------
# Dataset identity helpers
# ----------------------------

def get_dataset_id(dataset):
    """Return dataset ID as a string, even if source JSON has an int."""
    dataset_id = (
        dataset.get("id")
        or dataset.get("name")
        or "unknown-dataset"
    )

    return str(dataset_id)


def get_dataset_title(dataset):
    return str(dataset.get("title") or dataset.get("name") or get_dataset_id(dataset))


def get_tags(dataset):
    """
    Return tags for filtering.
    Uses dataset['tags'] if present and also includes dataset['keywords'].
    Tags are still filterable, but no longer shown as a visible table column.
    """
    tags = dataset.get("tags", [])
    keywords = dataset.get("keywords", [])

    if not isinstance(tags, list):
        tags = [tags]

    if not isinstance(keywords, list):
        keywords = [keywords]

    combined = tags + keywords

    clean_tags = sorted(
        {
            str(tag).strip()
            for tag in combined
            if tag is not None and str(tag).strip() != ""
        },
        key=lambda tag: tag.lower()
    )

    return clean_tags


# ----------------------------
# Curation helpers
# ----------------------------

def get_curation(dataset):
    curation = dataset.get("curation", {})

    if isinstance(curation, dict):
        return curation

    return {}


def get_collection_object(dataset):
    curation = get_curation(dataset)
    collection = curation.get("collection", {})

    if isinstance(collection, dict):
        return collection

    return {}


def get_collection_name(dataset):
    """
    Prefer curation.collection.name.
    Then curation.name.
    Then top-level collection.
    Return NA if no collection exists.
    """
    curation = get_curation(dataset)
    collection_object = get_collection_object(dataset)

    if collection_object.get("name"):
        return str(collection_object.get("name"))

    if curation.get("name"):
        return str(curation.get("name"))

    collection = dataset.get("collection")

    if collection is None:
        return "NA"

    if isinstance(collection, list):
        if not collection:
            return "NA"
        return ", ".join(str(item) for item in collection)

    if str(collection).strip() == "":
        return "NA"

    return str(collection)


def get_dataset_version(dataset):
    """
    Prefer curation.dataset_version.
    Then top-level dataset_version.
    Then newest value from all_versions.
    """
    curation = get_curation(dataset)

    if curation.get("dataset_version"):
        return str(curation.get("dataset_version"))

    if dataset.get("dataset_version"):
        return str(dataset.get("dataset_version"))

    all_versions = dataset.get("all_versions", [])

    if isinstance(all_versions, list) and all_versions:
        return str(sorted(all_versions, key=version_key, reverse=True)[0])

    return "TBD"


def get_release_version(dataset):
    """
    Prefer curation.release_version.
    Then newest value from all_releases.
    Then latest dataset release record.
    """
    curation = get_curation(dataset)

    if curation.get("release_version"):
        return str(curation.get("release_version"))

    all_releases = dataset.get("all_releases", [])

    if isinstance(all_releases, list) and all_releases:
        return str(sorted(all_releases, key=version_key, reverse=True)[0])

    releases = dataset.get("releases", {})

    if isinstance(releases, dict) and releases:
        return str(sorted(releases.keys(), key=version_key, reverse=True)[0])

    return "TBD"


def get_collection_version(dataset):
    """
    Prefer curation.collection.version.
    Then curation.collection_version.
    Return NA if not in a collection.
    """
    curation = get_curation(dataset)
    collection_object = get_collection_object(dataset)

    if collection_object.get("version"):
        return str(collection_object.get("version"))

    if curation.get("collection_version"):
        return str(curation.get("collection_version"))

    return "NA"


def get_collection_doi(dataset):
    collection_object = get_collection_object(dataset)

    if collection_object.get("collection_doi"):
        return collection_object.get("collection_doi")

    return ""


def get_curation_release_history(dataset):
    """
    Gets curation.releases, for example:
    {
      "v3.1.0": {
        "collection_version": "v3.1.0",
        "release_version": "v4.0.0",
        "collection_version_doi": "10.5281/zenodo.17860778"
      }
    }
    """
    curation = get_curation(dataset)
    releases = curation.get("releases", {})

    if isinstance(releases, dict):
        return releases

    return {}


def get_dataset_release_history(dataset):
    releases = dataset.get("releases", {})

    if isinstance(releases, dict):
        return releases

    return {}


def get_all_releases(dataset):
    all_releases = dataset.get("all_releases", [])

    if isinstance(all_releases, list):
        return [str(item) for item in all_releases]

    return []


def get_all_versions(dataset):
    all_versions = dataset.get("all_versions", [])

    if isinstance(all_versions, list):
        return [str(item) for item in all_versions]

    return []


# ----------------------------
# Load datasets.json and dataset-specific JSON
# ----------------------------

def find_dataset_detail_json(dataset_name):
    """
    Look for dataset-specific JSON using the dataset name.

    Supports:
    datasets/<dataset_name>.json
    datasets/<dataset_name>/<dataset_name>.json
    datasets/<dataset_name>/dataset.json
    """
    candidates = [
        DATASET_DIR / f"{dataset_name}.json",
        DATASET_DIR / dataset_name / f"{dataset_name}.json",
        DATASET_DIR / dataset_name / "dataset.json",
    ]

    for candidate in candidates:
        if candidate.exists():
            return candidate

    return None


def normalize_dataset_index(data):
    """
    Supports datasets.json shaped like:

    {
      "hafler-pmdbs-sn-rnaseq-pfc": {
        "name": "hafler-pmdbs-sn-rnaseq-pfc",
        ...
      }
    }

    or:

    [
      {
        "name": "hafler-pmdbs-sn-rnaseq-pfc",
        ...
      }
    ]
    """
    records = []

    if isinstance(data, dict):
        for key, value in data.items():
            if isinstance(value, dict):
                record = dict(value)
                record.setdefault("id", key)
                record.setdefault("name", key)
                records.append(record)

    elif isinstance(data, list):
        for item in data:
            if isinstance(item, dict):
                record = dict(item)
                record.setdefault("id", record.get("name", "unknown-dataset"))
                records.append(record)

    return records


def load_datasets_from_index_and_details():
    """
    Load datasets.json first, then enrich each record with its dataset-specific JSON.
    """
    index_data = load_json_file(DATASET_INDEX_FILE)
    index_records = normalize_dataset_index(index_data)

    datasets = []

    for index_record in index_records:
        dataset_name = get_dataset_id(index_record)
        detail_file = find_dataset_detail_json(dataset_name)

        if detail_file:
            try:
                detail_record = load_json_file(detail_file)

                # If detail JSON is shaped like {"dataset-name": {...}}, unwrap it.
                if (
                    isinstance(detail_record, dict)
                    and dataset_name in detail_record
                    and isinstance(detail_record[dataset_name], dict)
                ):
                    detail_record = detail_record[dataset_name]

                if isinstance(detail_record, dict):
                    merged_record = deep_merge(index_record, detail_record)
                    merged_record["_detail_file"] = str(detail_file)
                else:
                    merged_record = index_record
                    merged_record["_detail_file"] = ""

            except Exception as error:
                print(f"Could not load detail JSON for {dataset_name}: {error}")
                merged_record = index_record
                merged_record["_detail_file"] = ""

        else:
            print(f"No detail JSON found for {dataset_name}")
            merged_record = index_record
            merged_record["_detail_file"] = ""

        merged_record["_index_file"] = str(DATASET_INDEX_FILE)
        datasets.append(merged_record)

    return datasets


# ----------------------------
# Load dataset records
# ----------------------------

datasets = load_datasets_from_index_and_details()

if not datasets:
    raise ValueError("No dataset records found from datasets.json.")

# Deduplicate by dataset ID
unique_datasets = {}

for dataset in datasets:
    dataset_id = get_dataset_id(dataset)

    if dataset_id not in unique_datasets:
        unique_datasets[dataset_id] = dataset

datasets = sorted(
    unique_datasets.values(),
    key=lambda d: str(get_dataset_id(d)).lower()
)

all_tags = sorted(
    {
        tag
        for dataset in datasets
        for tag in get_tags(dataset)
    },
    key=lambda tag: tag.lower()
)

print(f"Loaded {len(datasets)} unique datasets")
print(f"Loaded {len(all_tags)} unique tags")


# ----------------------------
# Generate datasets.md
# ----------------------------

tag_options = ['<option value="">All tags</option>']

for tag in all_tags:
    tag_options.append(
        f'<option value="{esc(tag.lower())}">{esc(tag)}</option>'
    )

lines = [
    "# CRN Cloud Datasets",
    "",
    "Use this table to find dataset records, review curation details, and locate related release information.",
    "",
    '<div class="dataset-filters">',
    '  <input id="datasetSearch" class="dataset-search" type="text" placeholder="Filter by dataset, title, collection, release, collection version, DOI, CDE version, tag, or bucket path...">',
    '  <select id="tagFilter" class="tag-filter">',
    *tag_options,
    "  </select>",
    "</div>",
    "",
    '<p id="datasetCount" class="dataset-count"></p>',
    "",
    "<style>",
    ".md-grid {",
    "  max-width: 76rem;",
    "}",
    ".dataset-filters {",
    "  display: flex;",
    "  gap: 0.75rem;",
    "  align-items: center;",
    "  margin: 1rem 0 0.5rem 0;",
    "}",
    ".dataset-search {",
    "  flex: 1;",
    "  padding: 0.65rem;",
    "  border: 1px solid var(--md-default-fg-color--lightest);",
    "  border-radius: 0.45rem;",
    "  font-size: 0.9rem;",
    "}",
    ".tag-filter {",
    "  min-width: 180px;",
    "  padding: 0.65rem;",
    "  border: 1px solid var(--md-default-fg-color--lightest);",
    "  border-radius: 0.45rem;",
    "  font-size: 0.9rem;",
    "  background: var(--md-default-bg-color);",
    "  color: var(--md-default-fg-color);",
    "}",
    "@media (max-width: 700px) {",
    "  .dataset-filters {",
    "    flex-direction: column;",
    "    align-items: stretch;",
    "  }",
    "  .tag-filter {",
    "    width: 100%;",
    "  }",
    "}",
    ".dataset-count {",
    "  margin: 0 0 1rem 0;",
    "  color: var(--md-default-fg-color--light);",
    "}",
    ".dataset-table {",
    "  width: 100%;",
    "  border-collapse: collapse;",
    "  font-size: 0.76rem;",
    "  table-layout: fixed;",
    "}",
    ".dataset-table th, .dataset-table td {",
    "  border-bottom: 1px solid var(--md-default-fg-color--lightest);",
    "  padding: 0.4rem;",
    "  text-align: left;",
    "  vertical-align: top;",
    "  word-break: break-word;",
    "}",
    ".dataset-table th {",
    "  font-weight: 700;",
    "}",
    ".dataset-table th:nth-child(1), .dataset-table td:nth-child(1) {",
    "  width: 22%;",
    "}",
    ".dataset-table th:nth-child(2), .dataset-table td:nth-child(2) {",
    "  width: 31%;",
    "}",
    ".dataset-table th:nth-child(3), .dataset-table td:nth-child(3) {",
    "  width: 14%;",
    "}",
    ".dataset-table th:nth-child(4), .dataset-table td:nth-child(4) {",
    "  width: 9%;",
    "}",
    ".dataset-table th:nth-child(5), .dataset-table td:nth-child(5) {",
    "  width: 9%;",
    "}",
    ".dataset-table th:nth-child(6), .dataset-table td:nth-child(6) {",
    "  width: 10%;",
    "}",
    ".dataset-table th:nth-child(7), .dataset-table td:nth-child(7) {",
    "  width: 5%;",
    "}",
    ".dataset-toggle {",
    "  border: 1px solid var(--md-default-fg-color--lightest);",
    "  border-radius: 0.35rem;",
    "  padding: 0.25rem 0.45rem;",
    "  background: var(--md-default-bg-color);",
    "  cursor: pointer;",
    "  font-size: 0.72rem;",
    "}",
    ".dataset-toggle:hover {",
    "  border-color: var(--md-accent-fg-color);",
    "}",
    ".dataset-detail-row {",
    "  display: none;",
    "}",
    ".dataset-detail {",
    "  padding: 0.75rem;",
    "  border-left: 3px solid var(--md-accent-fg-color);",
    "  background: var(--md-code-bg-color);",
    "}",
    ".dataset-detail h4 {",
    "  margin-top: 0.75rem;",
    "  margin-bottom: 0.35rem;",
    "}",
    ".tag-pill {",
    "  display: inline-block;",
    "  padding: 0.12rem 0.4rem;",
    "  margin: 0.1rem 0.15rem 0.1rem 0;",
    "  border-radius: 999px;",
    "  background: var(--md-default-bg-color);",
    "  font-size: 0.72rem;",
    "  white-space: nowrap;",
    "}",
    ".mini-table {",
    "  width: 100%;",
    "  border-collapse: collapse;",
    "  font-size: 0.82rem;",
    "}",
    ".mini-table th, .mini-table td {",
    "  border-bottom: 1px solid var(--md-default-fg-color--lightest);",
    "  padding: 0.4rem;",
    "  text-align: left;",
    "  vertical-align: top;",
    "}",
    "</style>",
    "",
    '<table class="dataset-table" id="datasetTable">',
    "  <thead>",
    "    <tr>",
    "      <th>Dataset</th>",
    "      <th>Title</th>",
    "      <th>Collection</th>",
    "      <th>Dataset version</th>",
    "      <th>Release</th>",
    "      <th>Collection version</th>",
    "      <th>Details</th>",
    "    </tr>",
    "  </thead>",
    "  <tbody>",
]


for index, dataset in enumerate(datasets):
    dataset_id = get_dataset_id(dataset)
    dataset_title = get_dataset_title(dataset)
    description = str(dataset.get("description", ""))
    license_value = str(dataset.get("license", "TBD"))
    keywords = dataset.get("keywords", [])
    buckets = dataset.get("buckets", {})
    dataset_doi = dataset.get("doi", "")

    if not isinstance(keywords, list):
        keywords = [keywords]

    keywords = [str(keyword) for keyword in keywords if keyword is not None]

    if not isinstance(buckets, dict):
        buckets = {}

    tags = get_tags(dataset)
    tags_text = ", ".join(tags) if tags else "NA"
    tags_search = "||".join(tag.lower() for tag in tags)

    tags_html = (
        " ".join(f'<span class="tag-pill">{esc(tag)}</span>' for tag in tags)
        if tags
        else "NA"
    )

    collection_name = get_collection_name(dataset)
    dataset_version = get_dataset_version(dataset)
    release_version = get_release_version(dataset)
    collection_version = get_collection_version(dataset)
    collection_doi = get_collection_doi(dataset)
    curation_release_history = get_curation_release_history(dataset)
    dataset_release_history = get_dataset_release_history(dataset)
    all_releases = get_all_releases(dataset)
    all_versions = get_all_versions(dataset)

    detail_id = f"dataset-detail-{safe_id(dataset_id)}-{index}"

    curation_release_search = " ".join(
        [
            f"{release_key} {release_info.get('collection_version', '')} {release_info.get('release_version', '')} {release_info.get('collection_version_doi', '')}"
            for release_key, release_info in curation_release_history.items()
            if isinstance(release_info, dict)
        ]
    )

    dataset_release_search = " ".join(
        [
            f"{release_key} {release_info.get('dataset_version', '')} {release_info.get('cde_version', '')}"
            for release_key, release_info in dataset_release_history.items()
            if isinstance(release_info, dict)
        ]
    )

    search_text = " ".join([
        str(dataset_id),
        str(dataset_title),
        str(description),
        str(collection_name),
        str(dataset_version),
        str(release_version),
        str(collection_version),
        str(collection_doi),
        str(dataset_doi),
        " ".join(tags),
        " ".join(keywords),
        " ".join(all_releases),
        " ".join(all_versions),
        " ".join(str(value) for value in buckets.values()),
        curation_release_search,
        dataset_release_search,
    ]).lower()

    lines.extend([
        f'    <tr class="dataset-row" data-detail="{esc(detail_id)}" data-search="{esc(search_text)}" data-tags="{esc(tags_search)}">',
        f"      <td><code>{esc(dataset_id)}</code></td>",
        f"      <td>{esc(dataset_title)}</td>",
        f"      <td>{esc(collection_name)}</td>",
        f"      <td>{esc(dataset_version)}</td>",
        f"      <td>{esc(release_version)}</td>",
        f"      <td>{esc(collection_version)}</td>",
        f'      <td><button class="dataset-toggle" data-target="{esc(detail_id)}">View</button></td>',
        "    </tr>",
        f'    <tr id="{esc(detail_id)}" class="dataset-detail-row">',
        '      <td colspan="7">',
        '        <div class="dataset-detail">',
        f"          <h3>{esc(dataset_title)}</h3>",
        f"          <p><strong>Dataset ID:</strong> <code>{esc(dataset_id)}</code></p>",
        f"          <p><strong>Detail JSON:</strong> <code>{esc(dataset.get('_detail_file', ''))}</code></p>",
        f"          <p><strong>Description:</strong> {esc(description) if description else 'TBD'}</p>",
        f"          <p><strong>License:</strong> {esc(license_value)}</p>",
        f"          <p><strong>Dataset DOI:</strong> {doi_link(dataset_doi)}</p>",
        f"          <p><strong>Keywords:</strong> {esc(', '.join(keywords)) if keywords else 'TBD'}</p>",
        f"          <p><strong>Tags:</strong> {tags_html}</p>",
        "          <h4>Curation details</h4>",
        '          <table class="mini-table">',
        "            <thead>",
        "              <tr>",
        "                <th>Field</th>",
        "                <th>Value</th>",
        "              </tr>",
        "            </thead>",
        "            <tbody>",
        "              <tr>",
        "                <td>Collection</td>",
        f"                <td>{esc(collection_name)}</td>",
        "              </tr>",
        "              <tr>",
        "                <td>Dataset version</td>",
        f"                <td>{esc(dataset_version)}</td>",
        "              </tr>",
        "              <tr>",
        "                <td>Release version</td>",
        f"                <td>{esc(release_version)}</td>",
        "              </tr>",
        "              <tr>",
        "                <td>Collection version</td>",
        f"                <td>{esc(collection_version)}</td>",
        "              </tr>",
        "              <tr>",
        "                <td>Collection DOI</td>",
        f"                <td>{doi_link(collection_doi)}</td>",
        "              </tr>",
        "              <tr>",
        "                <td>All releases</td>",
        f"                <td>{esc(', '.join(all_releases)) if all_releases else 'TBD'}</td>",
        "              </tr>",
        "              <tr>",
        "                <td>All dataset versions</td>",
        f"                <td>{esc(', '.join(all_versions)) if all_versions else 'TBD'}</td>",
        "              </tr>",
        "            </tbody>",
        "          </table>",
        "          <h4>Collection release history</h4>",
        '          <table class="mini-table">',
        "            <thead>",
        "              <tr>",
        "                <th>Collection version</th>",
        "                <th>Release version</th>",
        "                <th>Collection version DOI</th>",
        "              </tr>",
        "            </thead>",
        "            <tbody>",
    ])

    if curation_release_history:
        for collection_release_key in sorted(curation_release_history.keys(), key=version_key, reverse=True):
            release_info = curation_release_history.get(collection_release_key, {})

            if not isinstance(release_info, dict):
                release_info = {}

            lines.extend([
                "              <tr>",
                f"                <td>{esc(release_info.get('collection_version', collection_release_key))}</td>",
                f"                <td>{esc(release_info.get('release_version', 'TBD'))}</td>",
                f"                <td>{doi_link(release_info.get('collection_version_doi', ''))}</td>",
                "              </tr>",
            ])
    else:
        lines.extend([
            "              <tr>",
            '                <td colspan="3">No collection release history listed.</td>',
            "              </tr>",
        ])

    lines.extend([
        "            </tbody>",
        "          </table>",
        "          <h4>Dataset release history</h4>",
        '          <table class="mini-table">',
        "            <thead>",
        "              <tr>",
        "                <th>Release</th>",
        "                <th>Dataset version</th>",
        "                <th>CDE version</th>",
        "              </tr>",
        "            </thead>",
        "            <tbody>",
    ])

    if dataset_release_history:
        for release_key in sorted(dataset_release_history.keys(), key=version_key, reverse=True):
            release_info = dataset_release_history.get(release_key, {})

            if not isinstance(release_info, dict):
                release_info = {}

            lines.extend([
                "              <tr>",
                f"                <td>{esc(release_key)}</td>",
                f"                <td>{esc(release_info.get('dataset_version', 'TBD'))}</td>",
                f"                <td>{esc(release_info.get('cde_version', 'TBD'))}</td>",
                "              </tr>",
            ])
    else:
        lines.extend([
            "              <tr>",
            '                <td colspan="3">No dataset release history listed.</td>',
            "              </tr>",
        ])

    lines.extend([
        "            </tbody>",
        "          </table>",
        "          <h4>Bucket paths</h4>",
        '          <table class="mini-table">',
        "            <thead>",
        "              <tr>",
        "                <th>Environment</th>",
        "                <th>Bucket path</th>",
        "              </tr>",
        "            </thead>",
        "            <tbody>",
    ])

    if buckets:
        for environment, bucket_path in buckets.items():
            lines.extend([
                "              <tr>",
                f"                <td>{esc(environment)}</td>",
                f"                <td><code>{esc(bucket_path)}</code></td>",
                "              </tr>",
            ])
    else:
        lines.extend([
            "              <tr>",
            '                <td colspan="2">No bucket paths listed.</td>',
            "              </tr>",
        ])

    lines.extend([
        "            </tbody>",
        "          </table>",
        "        </div>",
        "      </td>",
        "    </tr>",
    ])

lines.extend([
    "  </tbody>",
    "</table>",
    "",
])

markdown_text = "\n".join(lines)

OUT_FILE.parent.mkdir(parents=True, exist_ok=True)
OUT_FILE.write_text(markdown_text, encoding="utf-8")


# ----------------------------
# Generate JavaScript separately
# ----------------------------

js_text = """
function initializeDatasetPage() {
  const table = document.getElementById("datasetTable");

  if (!table) {
    return;
  }

  if (table.getAttribute("data-initialized") === "true") {
    return;
  }

  table.setAttribute("data-initialized", "true");

  const searchInput = document.getElementById("datasetSearch");
  const tagFilter = document.getElementById("tagFilter");
  const datasetCount = document.getElementById("datasetCount");
  const rows = Array.from(document.querySelectorAll(".dataset-row"));
  const buttons = Array.from(document.querySelectorAll(".dataset-toggle"));

  if (!rows.length) {
    return;
  }

  function updateCount(visibleCount) {
    if (datasetCount) {
      datasetCount.textContent = visibleCount + " of " + rows.length + " datasets shown";
    }
  }

  function closeDetailRow(row) {
    const detailId = row.getAttribute("data-detail");
    const detailRow = document.getElementById(detailId);
    const button = row.querySelector(".dataset-toggle");

    if (detailRow) {
      detailRow.style.display = "none";
    }

    if (button) {
      button.textContent = "View";
    }
  }

  function applyFilters() {
    const query = searchInput ? searchInput.value.toLowerCase().trim() : "";
    const selectedTag = tagFilter ? tagFilter.value.toLowerCase().trim() : "";

    let visibleCount = 0;

    rows.forEach(function (row) {
      const text = row.getAttribute("data-search") || "";
      const tags = row.getAttribute("data-tags") || "";

      const tagList = tags
        .split("||")
        .map(function (tag) {
          return tag.trim();
        })
        .filter(Boolean);

      const matchesText = query === "" || text.includes(query);
      const matchesTag = selectedTag === "" || tagList.includes(selectedTag);

      const isVisible = matchesText && matchesTag;

      row.style.display = isVisible ? "table-row" : "none";

      if (!isVisible) {
        closeDetailRow(row);
      }

      if (isVisible) {
        visibleCount += 1;
      }
    });

    updateCount(visibleCount);
  }

  buttons.forEach(function (button) {
    button.addEventListener("click", function () {
      const targetId = button.getAttribute("data-target");
      const detailRow = document.getElementById(targetId);

      if (!detailRow) {
        return;
      }

      const isOpen = detailRow.style.display === "table-row";
      detailRow.style.display = isOpen ? "none" : "table-row";
      button.textContent = isOpen ? "View" : "Hide";
    });
  });

  if (searchInput) {
    searchInput.addEventListener("input", applyFilters);
  }

  if (tagFilter) {
    tagFilter.addEventListener("change", applyFilters);
  }

  updateCount(rows.length);
}

if (typeof document$ !== "undefined") {
  document$.subscribe(function () {
    initializeDatasetPage();
  });
} else {
  document.addEventListener("DOMContentLoaded", initializeDatasetPage);
}
"""

JS_FILE.parent.mkdir(parents=True, exist_ok=True)
JS_FILE.write_text(js_text.strip() + "\n", encoding="utf-8")

print(f"Wrote: {OUT_FILE}")
print(f"Wrote: {JS_FILE}")

Learning Lab root: /Users/amaraalexander/Documents/GitHub/asap-crn-learning-lab
Dataset repo: /Users/amaraalexander/Documents/GitHub/cloud-datasets
Dataset index file: /Users/amaraalexander/Documents/GitHub/cloud-datasets/datasets.json
Dataset detail folder: /Users/amaraalexander/Documents/GitHub/cloud-datasets/datasets
Output Markdown: /Users/amaraalexander/Documents/GitHub/asap-crn-learning-lab/docs/rosetta-stone/datasets.md
Output JavaScript: /Users/amaraalexander/Documents/GitHub/asap-crn-learning-lab/docs/javascripts/dataset-filter.js
Loaded 61 unique datasets
Loaded 48 unique tags
Wrote: /Users/amaraalexander/Documents/GitHub/asap-crn-learning-lab/docs/rosetta-stone/datasets.md
Wrote: /Users/amaraalexander/Documents/GitHub/asap-crn-learning-lab/docs/javascripts/dataset-filter.js


In [ ]:
def normalize_dataset_records(data, source_file):
    records = []

    # Shape:
    # {
    #   "hafler-pmdbs-sn-rnaseq-pfc": {...},
    #   "another-dataset": {...}
    # }
    if isinstance(data, dict) and not any(
        key in data for key in ["name", "title", "description", "collection", "releases", "buckets"]
    ):
        for key, value in data.items():
            if isinstance(value, dict):
                record = dict(value)
                record.setdefault("id", key)
                record.setdefault("name", key)
                record["_source_file"] = str(source_file)
                records.append(record)

    return records

In [4]:
from pathlib import Path
import json
import re
import html


# ----------------------------
# Find Learning Lab repo root
# ----------------------------

def find_repo_root(start_path):
    start_path = Path(start_path).resolve()
    for path in [start_path] + list(start_path.parents):
        if (path / "mkdocs.yml").exists():
            return path
    raise FileNotFoundError("Could not find mkdocs.yml. Run this from inside the Learning Lab repo.")


ROOT = find_repo_root(Path.cwd())

DATASET_REPO         = ROOT.parent / "cloud-datasets"
COLLECTIONS_REPO     = ROOT.parent / "cloud-collections"
DATASET_INDEX_FILE   = DATASET_REPO / "datasets.json"
DATASET_DIR          = DATASET_REPO / "datasets"
COLLECTIONS_FILE     = COLLECTIONS_REPO / "collections.json"

if not DATASET_INDEX_FILE.exists():
    raise FileNotFoundError(f"Could not find dataset index file: {DATASET_INDEX_FILE}")
if not DATASET_DIR.exists():
    raise FileNotFoundError(f"Could not find dataset detail folder: {DATASET_DIR}")

OUT_FILE = ROOT / "docs" / "rosetta-stone" / "datasets.md"
JS_FILE  = ROOT / "docs" / "javascripts" / "dataset-filter.js"

print("Learning Lab root:", ROOT)
print("Dataset repo:", DATASET_REPO)
print("Collections repo:", COLLECTIONS_REPO)
print("Dataset index file:", DATASET_INDEX_FILE)
print("Dataset detail folder:", DATASET_DIR)
print("Collections file:", COLLECTIONS_FILE)
print("Output Markdown:", OUT_FILE)
print("Output JavaScript:", JS_FILE)


# ----------------------------
# Helpers
# ----------------------------

def esc(value):
    if value is None:
        return ""
    return html.escape(str(value), quote=True)


def safe_id(value):
    value = str(value).lower().strip()
    value = re.sub(r"[^a-z0-9]+", "-", value)
    return value.strip("-") or "dataset"


def doi_url(doi):
    if not doi:
        return ""
    doi = str(doi).strip()
    if doi.startswith("http://") or doi.startswith("https://"):
        return doi
    return f"https://doi.org/{doi}"


def doi_link(doi):
    if not doi:
        return ""
    doi = str(doi).strip()
    url = doi_url(doi)
    label = doi.replace("https://doi.org/", "").replace("http://doi.org/", "")
    return f'<a href="{esc(url)}" target="_blank" rel="noopener">{esc(label)} ↗</a>'


def doi_link_or_na(doi, missing="—"):
    result = doi_link(doi)
    return result if result else f'<span class="ds-na">{missing}</span>'


def version_key(version):
    numbers = re.findall(r"\d+", str(version))
    if not numbers:
        return (0,)
    return tuple(int(n) for n in numbers)


def load_json_file(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def deep_merge(base, extra):
    merged = dict(base)
    for key, value in extra.items():
        if key in merged and isinstance(merged[key], dict) and isinstance(value, dict):
            merged[key] = deep_merge(merged[key], value)
        else:
            merged[key] = value
    return merged


# ----------------------------
# Load cloud-collections/collections.json
# ----------------------------

def load_collections_index():
    """
    Load collections.json from the cloud-collections repo.
    Returns a dict keyed by collection name, or {} if the file is missing.

    Expected shape per entry:
    {
      "name": "pmdbs-sc-rnaseq",
      "title": "PMDBS scRNAseq",
      "collection_doi": "10.5281/zenodo.14373047",
      "current_version": "v3.1.0",
      "doi": "10.5281/zenodo.17860778",
      "datasets": [...],
      "release": {"version": "v4.0.0", "cde_version": "v3.3", "date": "..."},
      "versions": {
        "v3.1.0": {
          "doi": "10.5281/zenodo.17860778",
          "release": {"version": "v4.0.0", ...},
          ...
        },
        ...
      }
    }
    """
    if not COLLECTIONS_FILE.exists():
        print(f"WARNING: Collections file not found: {COLLECTIONS_FILE}")
        return {}

    try:
        data = load_json_file(COLLECTIONS_FILE)
    except Exception as e:
        print(f"WARNING: Could not load collections file: {e}")
        return {}

    # Normalise: accept both a list and a dict keyed by name
    if isinstance(data, dict):
        result = {}
        for key, value in data.items():
            if isinstance(value, dict):
                record = dict(value)
                record.setdefault("name", key)
                result[key] = record
        return result

    if isinstance(data, list):
        result = {}
        for item in data:
            if isinstance(item, dict) and item.get("name"):
                result[item["name"]] = item
        return result

    return {}


COLLECTIONS_INDEX = load_collections_index()
print(f"Loaded {len(COLLECTIONS_INDEX)} collections")


def get_collection_record(collection_name):
    """Return the collections.json entry for a collection name, or {}."""
    if not collection_name or collection_name == "NA":
        return {}
    return COLLECTIONS_INDEX.get(str(collection_name).strip(), {})


def get_collection_version_doi(collection_name, collection_version):
    """
    Look up the DOI for a specific collection version from collections.json.
    Returns empty string if not found.
    """
    record = get_collection_record(collection_name)
    if not record:
        return ""
    versions = record.get("versions", {})
    if not isinstance(versions, dict):
        return ""
    ver_entry = versions.get(str(collection_version).strip(), {})
    if not isinstance(ver_entry, dict):
        return ""
    return str(ver_entry.get("doi", "") or "")


def get_collection_top_doi(collection_name):
    """
    Return the top-level collection DOI from collections.json.
    Prefers collection_doi, falls back to doi.
    """
    record = get_collection_record(collection_name)
    if not record:
        return ""
    return str(record.get("collection_doi", "") or record.get("doi", "") or "")


# ----------------------------
# Dataset identity helpers
# ----------------------------

def get_dataset_id(dataset):
    return str(dataset.get("id") or dataset.get("name") or "unknown-dataset")


def get_dataset_title(dataset):
    return str(dataset.get("title") or dataset.get("name") or get_dataset_id(dataset))


def get_tags(dataset):
    tags     = dataset.get("tags", [])
    keywords = dataset.get("keywords", [])
    if not isinstance(tags, list):     tags     = [tags]
    if not isinstance(keywords, list): keywords = [keywords]
    combined = tags + keywords
    return sorted(
        {str(t).strip() for t in combined if t is not None and str(t).strip()},
        key=lambda t: t.lower()
    )


# ----------------------------
# Curation helpers
# ----------------------------

def get_curation(dataset):
    curation = dataset.get("curation", {})
    return curation if isinstance(curation, dict) else {}


def get_collection_object(dataset):
    collection = get_curation(dataset).get("collection", {})
    return collection if isinstance(collection, dict) else {}


def get_collection_name(dataset):
    curation        = get_curation(dataset)
    collection_obj  = get_collection_object(dataset)
    if collection_obj.get("name"): return str(collection_obj["name"])
    if curation.get("name"):       return str(curation["name"])
    collection = dataset.get("collection")
    if collection is None:                     return "NA"
    if isinstance(collection, list):           return ", ".join(str(i) for i in collection) if collection else "NA"
    return str(collection).strip() or "NA"


def get_dataset_version(dataset):
    curation = get_curation(dataset)
    if curation.get("dataset_version"): return str(curation["dataset_version"])
    if dataset.get("dataset_version"):  return str(dataset["dataset_version"])
    all_v = dataset.get("all_versions", [])
    if isinstance(all_v, list) and all_v:
        return str(sorted(all_v, key=version_key, reverse=True)[0])
    return "TBD"


def get_release_version(dataset):
    curation = get_curation(dataset)
    if curation.get("release_version"): return str(curation["release_version"])
    all_r = dataset.get("all_releases", [])
    if isinstance(all_r, list) and all_r:
        return str(sorted(all_r, key=version_key, reverse=True)[0])
    releases = dataset.get("releases", {})
    if isinstance(releases, dict) and releases:
        return str(sorted(releases.keys(), key=version_key, reverse=True)[0])
    return "TBD"


def get_collection_version(dataset):
    curation       = get_curation(dataset)
    collection_obj = get_collection_object(dataset)
    if collection_obj.get("version"):        return str(collection_obj["version"])
    if curation.get("collection_version"):   return str(curation["collection_version"])
    return "NA"


def get_collection_doi(dataset):
    """
    Return the collection DOI, enriched from collections.json when available.
    Falls back to what is in the dataset curation block.
    """
    collection_name = get_collection_name(dataset)

    # Try collections.json first
    top_doi = get_collection_top_doi(collection_name)
    if top_doi:
        return top_doi

    # Fall back to curation block
    collection_obj = get_collection_object(dataset)
    return str(collection_obj.get("collection_doi", "") or get_curation(dataset).get("collection_doi", "") or "")


def get_curation_release_history(dataset):
    releases = get_curation(dataset).get("releases", {})
    return releases if isinstance(releases, dict) else {}


def get_dataset_release_history(dataset):
    releases = dataset.get("releases", {})
    return releases if isinstance(releases, dict) else {}


def get_all_releases(dataset):
    return [str(i) for i in dataset.get("all_releases", []) if isinstance(dataset.get("all_releases", []), list)]


def get_all_versions(dataset):
    return [str(i) for i in dataset.get("all_versions", []) if isinstance(dataset.get("all_versions", []), list)]


# ----------------------------
# Curation status derivation
# ----------------------------

def has_curation(dataset):
    curation = dataset.get("curation")
    if not isinstance(curation, dict) or not curation:
        return False
    return bool(curation.get("dataset_version") or curation.get("release_version"))


def derive_curation_status(release_key, dataset, all_releases_sorted):
    """
    all_releases_sorted is newest-first.
    Returns: "added" | "updated" | "unchanged" | "not-curated"
    """
    if not has_curation(dataset):
        return "not-curated"

    curation         = dataset.get("curation", {})
    curation_release = curation.get("release_version", "")
    chronological    = list(reversed(all_releases_sorted))  # oldest→newest

    if version_key(release_key) < version_key(curation_release):
        return "not-curated"
    if release_key != curation_release:
        return "unchanged"

    # This IS the curation release — added vs updated
    current_index = chronological.index(release_key) if release_key in chronological else 0
    prior_releases = chronological[:current_index]
    cur_rel_hist   = get_curation_release_history(dataset)
    prior_curated  = [
        r for r in prior_releases
        if r in cur_rel_hist
        or any(isinstance(v, dict) and v.get("release_version") == r for v in cur_rel_hist.values())
    ]
    return "updated" if prior_curated else "added"


def curation_status_badge(status):
    configs = {
        "added":       ("added",       "#e8f5ee", "#0d6b3f", "#6fcf97", "#0d6b3f"),
        "updated":     ("updated",     "#e8f0fb", "#1a4fa0", "#7baee8", "#1a4fa0"),
        "unchanged":   ("unchanged",   "#f0eeea", "#5a5850", "#bbb9b0", "#5a5850"),
        "not-curated": ("not curated", "#fef3e2", "#8a5a00", "#f0b429", "#8a5a00"),
    }
    label, bg, color, border, dot_color = configs.get(status, configs["not-curated"])
    dot = (
        f'<span style="display:inline-block;width:6px;height:6px;border-radius:50%;'
        f'background:{dot_color};margin-right:4px;vertical-align:middle"></span>'
    )
    return (
        f'<span class="curation-badge curation-badge--{esc(status)}" '
        f'style="display:inline-flex;align-items:center;font-size:0.72rem;padding:2px 8px;'
        f'border-radius:3px;font-weight:500;white-space:nowrap;font-family:monospace;'
        f'background:{bg};color:{color};border:1px solid {border}">'
        f'{dot}{esc(label)}</span>'
    )


# ----------------------------
# Dataset loading
# ----------------------------

def find_dataset_detail_json(dataset_name):
    for candidate in [
        DATASET_DIR / f"{dataset_name}.json",
        DATASET_DIR / dataset_name / f"{dataset_name}.json",
        DATASET_DIR / dataset_name / "dataset.json",
    ]:
        if candidate.exists():
            return candidate
    return None


def normalize_dataset_index(data):
    records = []
    if isinstance(data, dict):
        for key, value in data.items():
            if isinstance(value, dict):
                rec = dict(value)
                rec.setdefault("id", key)
                rec.setdefault("name", key)
                records.append(rec)
    elif isinstance(data, list):
        for item in data:
            if isinstance(item, dict):
                rec = dict(item)
                rec.setdefault("id", rec.get("name", "unknown-dataset"))
                records.append(rec)
    return records


def load_datasets_from_index_and_details():
    index_records = normalize_dataset_index(load_json_file(DATASET_INDEX_FILE))
    datasets = []
    for index_record in index_records:
        dataset_name = get_dataset_id(index_record)
        detail_file  = find_dataset_detail_json(dataset_name)
        if detail_file:
            try:
                detail = load_json_file(detail_file)
                if isinstance(detail, dict) and dataset_name in detail and isinstance(detail[dataset_name], dict):
                    detail = detail[dataset_name]
                merged = deep_merge(index_record, detail) if isinstance(detail, dict) else index_record
                merged["_detail_file"] = str(detail_file)
            except Exception as e:
                print(f"Could not load detail JSON for {dataset_name}: {e}")
                merged = index_record
                merged["_detail_file"] = ""
        else:
            print(f"No detail JSON found for {dataset_name}")
            merged = index_record
            merged["_detail_file"] = ""
        merged["_index_file"] = str(DATASET_INDEX_FILE)
        datasets.append(merged)
    return datasets


datasets = load_datasets_from_index_and_details()
if not datasets:
    raise ValueError("No dataset records found from datasets.json.")

unique_datasets = {}
for d in datasets:
    did = get_dataset_id(d)
    if did not in unique_datasets:
        unique_datasets[did] = d

datasets = sorted(unique_datasets.values(), key=lambda d: str(get_dataset_id(d)).lower())

all_tags = sorted({tag for d in datasets for tag in get_tags(d)}, key=lambda t: t.lower())

# Collect unique filter values for the dropdowns
all_releases_values    = sorted(
    {get_release_version(d) for d in datasets if get_release_version(d) != "TBD"},
    key=version_key, reverse=True
)
all_cde_versions       = sorted(
    {
        ri.get("cde_version", "")
        for d in datasets
        for ri in get_dataset_release_history(d).values()
        if isinstance(ri, dict) and ri.get("cde_version")
    },
    key=version_key, reverse=True
)
all_collection_values  = sorted(
    {get_collection_name(d) for d in datasets if get_collection_name(d) not in ("NA", "", "TBD")},
    key=lambda s: s.lower()
)

print(f"Loaded {len(datasets)} unique datasets")
print(f"Loaded {len(all_tags)} unique tags")
print(f"Releases for filter: {all_releases_values}")
print(f"CDE versions for filter: {all_cde_versions}")
print(f"Collections for filter: {all_collection_values[:5]}...")


# ----------------------------
# HTML helpers — detail panel
# ----------------------------

def render_meta_strip(dataset_doi, license_value, collection_name, collection_doi, dataset_version):
    doi_cell     = doi_link(dataset_doi)     or "TBD"
    col_doi_cell = doi_link(collection_doi)  or "TBD"
    col_na       = "ds-meta-na" if collection_name == "NA" else ""
    cdoi_na      = "ds-meta-na" if not collection_doi   else ""
    return f"""
          <div class="ds-meta-strip">
            <div class="ds-meta-item">
              <span class="ds-meta-label">License</span>
              <span class="ds-meta-value">{esc(license_value)}</span>
            </div>
            <div class="ds-meta-item">
              <span class="ds-meta-label">Dataset DOI</span>
              <span class="ds-meta-value">{doi_cell}</span>
            </div>
            <div class="ds-meta-item">
              <span class="ds-meta-label">Collection</span>
              <span class="ds-meta-value {col_na}">{esc(collection_name)}</span>
            </div>
            <div class="ds-meta-item">
              <span class="ds-meta-label">Collection DOI</span>
              <span class="ds-meta-value {cdoi_na}">{col_doi_cell}</span>
            </div>
            <div class="ds-meta-item">
              <span class="ds-meta-label">Latest version</span>
              <span class="ds-meta-value ds-mono">{esc(dataset_version)}</span>
            </div>
          </div>"""


def render_release_tabs(releases_sorted, dataset, panel_id_prefix):
    """releases_sorted is newest-first."""
    tabs = []
    for i, rk in enumerate(releases_sorted):
        status  = derive_curation_status(rk, dataset, releases_sorted)
        badge   = curation_status_badge(status)
        is_latest   = (i == 0)
        latest_html = '<span class="ds-latest-badge">latest</span>' if is_latest else ""
        active      = " ds-rtab--active" if is_latest else ""
        tabs.append(
            f'<button class="ds-rtab{active}" '
            f'data-panel="{esc(panel_id_prefix)}-{esc(rk)}" '
            f'onclick="dsTabSwitch(this)">'
            f'<span class="ds-mono">{esc(rk)}</span>'
            f'{latest_html}{badge}'
            f'</button>'
        )
    return '<div class="ds-release-tabs">' + "".join(tabs) + "</div>"


def render_curation_card(release_key, dataset, status):
    """
    Curation details are FIXED for the whole dataset.
    Only the status badge changes per release tab.
    CDE version is looked up from releases[curation.release_version].
    Collection DOI is enriched from collections.json.
    """
    badge = curation_status_badge(status)

    if status == "not-curated":
        return f"""
              <div class="ds-card">
                <div class="ds-card-header">
                  <span class="ds-card-title">Curation details</span>
                  {badge}
                  <span class="ds-card-header-right ds-mono">{esc(release_key)}</span>
                </div>
                <div class="ds-no-curation">
                  <span class="ds-no-curation-icon">∅</span>
                  <p>No curation files were produced for this dataset in release {esc(release_key)}.</p>
                </div>
              </div>"""

    curation       = get_curation(dataset)
    collection_obj = get_collection_object(dataset)
    dataset_rel_history = get_dataset_release_history(dataset)

    dataset_ver      = esc(curation.get("dataset_version", "TBD"))
    release_ver      = esc(curation.get("release_version",  "TBD"))
    curation_rel_key = curation.get("release_version", "")
    cde_ver          = esc(dataset_rel_history.get(curation_rel_key, {}).get("cde_version") or "TBD")

    col_ver = collection_obj.get("version") or curation.get("collection_version") or None
    col_ver_cell = (
        f'<span class="ds-mono">{esc(col_ver)}</span>' if col_ver
        else '<span class="ds-na">—</span>'
    )

    col_name = collection_obj.get("name") or curation.get("name") or None
    col_name_cell = esc(col_name) if col_name else '<span class="ds-na">—</span>'

    # Enrich collection DOI from collections.json
    collection_name = get_collection_name(dataset)
    col_doi = get_collection_top_doi(collection_name)
    if not col_doi:
        col_doi = collection_obj.get("collection_doi") or curation.get("collection_doi") or ""

    # If we have a specific collection version, try to get its version DOI too
    col_ver_doi = ""
    if col_ver:
        col_ver_doi = get_collection_version_doi(collection_name, col_ver)

    col_doi_cell     = doi_link_or_na(col_doi,     "TBD")
    col_ver_doi_cell = doi_link_or_na(col_ver_doi, "—")

    return f"""
              <div class="ds-card">
                <div class="ds-card-header">
                  <span class="ds-card-title">Curation details</span>
                  {badge}
                  <span class="ds-card-header-right ds-mono">{esc(release_key)}</span>
                </div>
                <div class="ds-card-body">
                  <div class="ds-field"><span class="ds-field-label">Dataset version</span><span class="ds-field-value ds-highlight ds-mono">{dataset_ver}</span></div>
                  <div class="ds-field"><span class="ds-field-label">Release version</span><span class="ds-field-value ds-highlight ds-mono">{release_ver}</span></div>
                  <div class="ds-field"><span class="ds-field-label">CDE version</span><span class="ds-field-value ds-mono">{cde_ver}</span></div>
                  <div class="ds-field"><span class="ds-field-label">Collection</span><span class="ds-field-value">{col_name_cell}</span></div>
                  <div class="ds-field"><span class="ds-field-label">Collection version</span><span class="ds-field-value">{col_ver_cell}</span></div>
                  <div class="ds-field"><span class="ds-field-label">Collection DOI</span><span class="ds-field-value">{col_doi_cell}</span></div>
                  <div class="ds-field"><span class="ds-field-label">Collection version DOI</span><span class="ds-field-value">{col_ver_doi_cell}</span></div>
                </div>
              </div>"""


def render_bucket_card(release_key, buckets, status):
    # Only prod and raw are shown. UAT and DEV are excluded for all statuses.
    # Not-curated releases show raw only (no curated prod output exists).
    env_order  = ["prod", "raw"]
    env_colors = {"prod": "#0d6b3f", "raw": "#5a5850"}

    filtered = {k: v for k, v in buckets.items() if k in ("prod", "raw")}

    if not filtered:
        bucket_rows = '<div class="ds-no-buckets">No bucket paths listed.</div>'
    else:
        rows = []
        for env in sorted(filtered, key=lambda e: env_order.index(e) if e in env_order else 99):
            path  = filtered[env]
            color = env_colors.get(env, "#5a5850")
            rows.append(
                f'<div class="ds-env-row">'
                f'<span class="ds-env-dot" style="background:{color}"></span>'
                f'<span class="ds-env-name" style="color:{color}">{esc(env)}</span>'
                f'<span class="ds-env-path">{esc(path)}</span>'
                f'<button class="ds-copy-btn" data-path="{esc(path)}" onclick="dsCopyPath(this)">Copy</button>'
                f'</div>'
            )
        bucket_rows = "".join(rows)

    return f"""
              <div class="ds-card">
                <div class="ds-card-header">
                  <span class="ds-card-title">Bucket access</span>
                  <span class="ds-card-header-right ds-mono">{esc(release_key)}</span>
                </div>
                <div class="ds-card-body ds-bucket-body">
                  {bucket_rows}
                </div>
              </div>"""


def render_release_panels(releases_sorted, dataset, buckets, panel_id_prefix):
    panels = []
    for i, rk in enumerate(releases_sorted):
        status    = derive_curation_status(rk, dataset, releases_sorted)
        display   = "flex" if i == 0 else "none"
        panel_id  = f"{panel_id_prefix}-{rk}"
        panels.append(
            f'<div id="{esc(panel_id)}" class="ds-panel-sections" style="display:{display}">'
            f'{render_curation_card(rk, dataset, status)}'
            f'{render_bucket_card(rk, buckets, status)}'
            f'</div>'
        )
    return "".join(panels)


def render_full_history_table(releases_sorted, dataset, dataset_release_history, curation_release_history):
    collection_name = get_collection_name(dataset)
    rows = []
    for i, rk in enumerate(releases_sorted):
        rel_info    = dataset_release_history.get(rk, {})
        status      = derive_curation_status(rk, dataset, releases_sorted)
        badge       = curation_status_badge(status)
        dataset_ver = rel_info.get("dataset_version", "TBD")
        cde_ver     = rel_info.get("cde_version", "TBD")

        # Collection version: check curation_release_history first
        col_ver = "—"
        col_doi = ""
        for col_key, col_info in curation_release_history.items():
            if isinstance(col_info, dict) and col_info.get("release_version") == rk:
                col_ver = col_info.get("collection_version", col_key) or "—"
                # Try collections.json for the version DOI, fall back to curation history
                col_doi = get_collection_version_doi(collection_name, col_ver)
                if not col_doi:
                    col_doi = col_info.get("collection_version_doi", "")
                break

        col_doi_cell = doi_link_or_na(col_doi, "—")
        is_latest    = (i == 0)
        latest_html  = '<span class="ds-latest-badge">latest</span>' if is_latest else ""
        active_class = " ds-history-row--active" if is_latest else ""

        rows.append(
            f'<tr class="ds-history-row{active_class}" data-release="{esc(rk)}">'
            f'<td><span class="ds-mono">{esc(rk)}</span> {latest_html}</td>'
            f'<td><span class="ds-mono">{esc(dataset_ver)}</span></td>'
            f'<td><span class="ds-mono">{esc(col_ver)}</span></td>'
            f'<td>{col_doi_cell}</td>'
            f'<td><span class="ds-mono">{esc(cde_ver)}</span></td>'
            f'<td>{badge}</td>'
            f'</tr>'
        )

    empty = '<tr><td colspan="6">No release history listed.</td></tr>'
    return f"""
          <div class="ds-section">
            <div class="ds-section-header">Full release history</div>
            <div class="ds-card ds-card--table">
              <table class="mini-table">
                <thead>
                  <tr>
                    <th>Release</th>
                    <th>Dataset version</th>
                    <th>Collection version</th>
                    <th>Collection DOI</th>
                    <th>CDE version</th>
                    <th>Curation</th>
                  </tr>
                </thead>
                <tbody>{"".join(rows) if rows else empty}</tbody>
              </table>
            </div>
            <p class="ds-table-hint">Click a row to switch to that release above.</p>
          </div>"""


def render_collection_release_history(curation_release_history, collection_name):
    if not curation_release_history:
        return ""
    rows = []
    for col_key in sorted(curation_release_history.keys(), key=version_key, reverse=True):
        col_info = curation_release_history.get(col_key, {})
        if not isinstance(col_info, dict):
            col_info = {}
        col_ver = esc(col_info.get("collection_version", col_key))
        rel_ver = esc(col_info.get("release_version", "TBD"))
        # Enrich DOI from collections.json
        col_doi = get_collection_version_doi(collection_name, col_info.get("collection_version", col_key))
        if not col_doi:
            col_doi = col_info.get("collection_version_doi", "")
        rows.append(f"<tr><td>{col_ver}</td><td>{rel_ver}</td><td>{doi_link_or_na(col_doi, '—')}</td></tr>")

    return f"""
          <div class="ds-section">
            <div class="ds-section-header">Collection release history</div>
            <div class="ds-card ds-card--table">
              <table class="mini-table">
                <thead>
                  <tr><th>Collection version</th><th>Release version</th><th>Collection version DOI</th></tr>
                </thead>
                <tbody>{"".join(rows)}</tbody>
              </table>
            </div>
          </div>"""


# ----------------------------
# Build filter <select> options
# ----------------------------

def build_select(element_id, label, values, css_class="ds-filter-select"):
    options = [f'<option value="">All {label}</option>']
    for v in values:
        options.append(f'<option value="{esc(v)}">{esc(v)}</option>')
    return (
        f'<select id="{element_id}" class="{css_class}" '
        f'aria-label="Filter by {label}">'
        + "".join(options)
        + "</select>"
    )


# ----------------------------
# Generate datasets.md
# ----------------------------

tag_options = ['<option value="">All tags</option>']
for tag in all_tags:
    tag_options.append(f'<option value="{esc(tag.lower())}">{esc(tag)}</option>')

lines = [
    "# CRN Cloud Datasets",
    "",
    "Use this table to find dataset records, review curation details, and locate related release information.",
    "",
    # Filter bar — search + four dropdowns
    '<div class="dataset-filters">',
    '  <input id="datasetSearch" class="dataset-search" type="text"'
    '   placeholder="Filter by dataset, title, collection, release, CDE version, DOI, tag, or bucket path...">',
    '  ' + build_select("releaseFilter",     "releases",    all_releases_values),
    '  ' + build_select("cdeFilter",         "CDE versions", all_cde_versions),
    '  ' + build_select("collectionFilter",  "collections", all_collection_values),
    '  <select id="tagFilter" class="ds-filter-select" aria-label="Filter by tag">',
    *[f'    {o}' for o in tag_options],
    "  </select>",
    "</div>",
    "",
    '<p id="datasetCount" class="dataset-count"></p>',
    "",

    # ── Styles ──────────────────────────────────────────────────────────────
    "<style>",

    ".md-grid { max-width: 76rem; }",

    # Filter bar
    ".dataset-filters { display: flex; gap: 0.5rem; align-items: center; margin: 1rem 0 0.5rem; flex-wrap: wrap; }",
    ".dataset-search { flex: 2; min-width: 220px; padding: 0.6rem; border: 1px solid var(--md-default-fg-color--lightest); border-radius: 0.45rem; font-size: 0.85rem; }",
    ".ds-filter-select { flex: 1; min-width: 130px; padding: 0.6rem; border: 1px solid var(--md-default-fg-color--lightest); border-radius: 0.45rem; font-size: 0.85rem; background: var(--md-default-bg-color); color: var(--md-default-fg-color); }",
    "@media (max-width: 700px) { .dataset-filters { flex-direction: column; align-items: stretch; } }",
    ".dataset-count { margin: 0 0 0.75rem; color: var(--md-default-fg-color--light); font-size: 0.85rem; }",

    # Summary table
    ".dataset-table { width: 100%; border-collapse: collapse; font-size: 0.76rem; table-layout: fixed; }",
    ".dataset-table th, .dataset-table td { border-bottom: 1px solid var(--md-default-fg-color--lightest); padding: 0.4rem; text-align: left; vertical-align: top; word-break: break-word; }",
    ".dataset-table th { font-weight: 700; }",
    # Sortable header
    ".dataset-table th.sortable { cursor: pointer; user-select: none; white-space: nowrap; }",
    ".dataset-table th.sortable:hover { color: var(--md-accent-fg-color); }",
    ".sort-icon { display: inline-block; margin-left: 4px; opacity: 0.4; font-style: normal; font-size: 0.7rem; }",
    ".sort-icon.asc  { opacity: 1; }",
    ".sort-icon.desc { opacity: 1; }",
    # Column widths
    ".dataset-table th:nth-child(1), .dataset-table td:nth-child(1) { width: 21%; }",
    ".dataset-table th:nth-child(2), .dataset-table td:nth-child(2) { width: 27%; }",
    ".dataset-table th:nth-child(3), .dataset-table td:nth-child(3) { width: 13%; }",
    ".dataset-table th:nth-child(4), .dataset-table td:nth-child(4) { width: 9%; }",
    ".dataset-table th:nth-child(5), .dataset-table td:nth-child(5) { width: 9%; }",
    ".dataset-table th:nth-child(6), .dataset-table td:nth-child(6) { width: 12%; }",
    ".dataset-table th:nth-child(7), .dataset-table td:nth-child(7) { width: 9%; white-space: nowrap; }",

    ".dataset-toggle { border: 1px solid var(--md-default-fg-color--lightest); border-radius: 0.35rem; padding: 0.25rem 0.45rem; background: var(--md-default-bg-color); cursor: pointer; font-size: 0.72rem; }",
    ".dataset-toggle:hover { border-color: var(--md-accent-fg-color); }",
    ".dataset-detail-row { display: none; }",

    # Detail panel
    ".dataset-detail { padding: 1rem 1rem 1.5rem; border-left: 3px solid var(--md-accent-fg-color); background: var(--md-code-bg-color); }",
    ".ds-title { font-size: 1.05rem; font-weight: 600; margin-bottom: 0.2rem; line-height: 1.35; }",
    ".ds-id { font-family: monospace; font-size: 0.78rem; color: var(--md-accent-fg-color); margin-bottom: 0.5rem; }",
    ".ds-tag-row { display: flex; gap: 5px; flex-wrap: wrap; margin-bottom: 0.75rem; }",
    ".tag-pill { display: inline-block; padding: 0.12rem 0.45rem; border-radius: 3px; background: var(--md-default-bg-color); border: 1px solid var(--md-default-fg-color--lightest); font-size: 0.72rem; font-family: monospace; white-space: nowrap; }",

    # Meta strip
    ".ds-meta-strip { display: flex; gap: 0; border: 1px solid var(--md-default-fg-color--lightest); border-radius: 5px; overflow: hidden; margin-bottom: 1rem; }",
    ".ds-meta-item { flex: 1; padding: 0.45rem 0.65rem; border-right: 1px solid var(--md-default-fg-color--lightest); }",
    ".ds-meta-item:last-child { border-right: none; }",
    ".ds-meta-label { font-size: 0.65rem; font-weight: 600; letter-spacing: 0.06em; text-transform: uppercase; color: var(--md-default-fg-color--light); display: block; margin-bottom: 2px; }",
    ".ds-meta-value { font-size: 0.8rem; font-weight: 500; }",
    ".ds-meta-na { color: var(--md-default-fg-color--light); font-style: italic; font-weight: 400; }",

    # Section
    ".ds-section { margin-bottom: 1rem; }",
    ".ds-section-header { font-size: 0.68rem; font-weight: 600; letter-spacing: 0.08em; text-transform: uppercase; color: var(--md-default-fg-color--light); padding-bottom: 0.35rem; border-bottom: 1px solid var(--md-default-fg-color--lightest); margin-bottom: 0.6rem; }",
    ".ds-description { font-size: 0.82rem; color: var(--md-default-fg-color); line-height: 1.65; }",

    # Release tabs
    ".ds-release-tabs { display: flex; gap: 0; border: 1px solid var(--md-default-fg-color--lightest); border-radius: 5px; overflow: hidden; margin-bottom: 0.75rem; width: fit-content; flex-wrap: wrap; }",
    ".ds-rtab { font-size: 0.75rem; padding: 0.35rem 0.75rem; cursor: pointer; color: var(--md-default-fg-color--light); background: transparent; border: none; border-right: 1px solid var(--md-default-fg-color--lightest); display: inline-flex; align-items: center; gap: 6px; font-weight: 500; }",
    ".ds-rtab:last-child { border-right: none; }",
    ".ds-rtab:hover { background: var(--md-default-bg-color); color: var(--md-default-fg-color); }",
    ".ds-rtab--active { background: var(--md-default-fg-color); color: var(--md-default-bg-color); }",

    ".ds-panel-sections { display: flex; flex-direction: column; gap: 0.6rem; }",

    # Cards
    ".ds-card { border: 1px solid var(--md-default-fg-color--lightest); border-radius: 5px; overflow: hidden; background: var(--md-default-bg-color); }",
    ".ds-card--table { overflow-x: auto; }",
    ".ds-card-header { padding: 0.4rem 0.75rem; background: var(--md-code-bg-color); border-bottom: 1px solid var(--md-default-fg-color--lightest); display: flex; align-items: center; gap: 0.5rem; }",
    ".ds-card-title { font-size: 0.75rem; font-weight: 600; }",
    ".ds-card-header-right { margin-left: auto; font-size: 0.72rem; color: var(--md-default-fg-color--light); }",
    ".ds-card-body { padding: 0.5rem 0.75rem; }",

    # Fields
    ".ds-field { display: flex; justify-content: space-between; align-items: baseline; padding: 0.3rem 0; border-bottom: 1px solid var(--md-default-fg-color--lightest); gap: 0.5rem; }",
    ".ds-field:last-child { border-bottom: none; }",
    ".ds-field-label { font-size: 0.78rem; color: var(--md-default-fg-color--light); flex-shrink: 0; }",
    ".ds-field-value { font-size: 0.78rem; font-weight: 500; text-align: right; }",
    ".ds-highlight { color: var(--md-accent-fg-color); }",
    ".ds-mono { font-family: monospace; }",
    ".ds-na { color: var(--md-default-fg-color--light); font-style: italic; font-weight: 400; }",

    # No curation / no buckets
    ".ds-no-curation { padding: 1.2rem; text-align: center; color: var(--md-default-fg-color--light); }",
    ".ds-no-curation-icon { font-size: 1.4rem; display: block; margin-bottom: 0.35rem; }",
    ".ds-no-curation p { font-size: 0.8rem; }",
    ".ds-no-buckets { padding: 0.75rem; font-size: 0.8rem; color: var(--md-default-fg-color--light); }",

    # Bucket rows
    ".ds-bucket-body { padding: 0.25rem 0.75rem; }",
    ".ds-env-row { display: flex; align-items: center; gap: 0.5rem; padding: 0.4rem 0; border-bottom: 1px solid var(--md-default-fg-color--lightest); }",
    ".ds-env-row:last-child { border-bottom: none; }",
    ".ds-env-dot { width: 7px; height: 7px; border-radius: 50%; flex-shrink: 0; }",
    ".ds-env-name { font-size: 0.68rem; font-weight: 600; font-family: monospace; width: 34px; flex-shrink: 0; text-transform: uppercase; letter-spacing: 0.05em; }",
    ".ds-env-path { font-family: monospace; font-size: 0.72rem; color: var(--md-default-fg-color--light); flex: 1; min-width: 0; overflow: hidden; text-overflow: ellipsis; white-space: nowrap; }",
    ".ds-copy-btn { font-size: 0.65rem; padding: 2px 7px; border: 1px solid var(--md-default-fg-color--lightest); border-radius: 3px; cursor: pointer; background: var(--md-code-bg-color); flex-shrink: 0; }",
    ".ds-copy-btn:hover { border-color: var(--md-accent-fg-color); }",
    ".ds-copy-btn--copied { background: #e8f5ee; color: #0d6b3f; border-color: #6fcf97; }",

    # History table
    ".ds-history-row { cursor: pointer; }",
    ".ds-history-row:hover td { background: var(--md-code-bg-color); }",
    ".ds-history-row--active td { background: #e8f0fb; }",
    ".ds-table-hint { font-size: 0.7rem; color: var(--md-default-fg-color--light); margin-top: 0.35rem; }",
    ".ds-latest-badge { font-size: 0.65rem; padding: 1px 6px; border-radius: 20px; background: #e8f5ee; color: #0d6b3f; border: 1px solid #6fcf97; font-weight: 500; }",

    # Mini table
    ".mini-table { width: 100%; border-collapse: collapse; font-size: 0.82rem; }",
    ".mini-table th, .mini-table td { border-bottom: 1px solid var(--md-default-fg-color--lightest); padding: 0.4rem; text-align: left; vertical-align: top; }",

    "</style>",
    "",

    # Summary table — sortable headers carry data-col index
    '<table class="dataset-table" id="datasetTable">',
    "  <thead>",
    "    <tr>",
    '      <th class="sortable" data-col="0">Dataset <i class="sort-icon">⇅</i></th>',
    '      <th class="sortable" data-col="1">Title <i class="sort-icon">⇅</i></th>',
    '      <th class="sortable" data-col="2">Collection <i class="sort-icon">⇅</i></th>',
    '      <th class="sortable" data-col="3">Dataset version <i class="sort-icon">⇅</i></th>',
    '      <th class="sortable" data-col="4">Release <i class="sort-icon">⇅</i></th>',
    '      <th class="sortable" data-col="5">Collection version <i class="sort-icon">⇅</i></th>',
    "      <th>Details</th>",
    "    </tr>",
    "  </thead>",
    "  <tbody>",
]


for index, dataset in enumerate(datasets):
    dataset_id      = get_dataset_id(dataset)
    dataset_title   = get_dataset_title(dataset)
    description     = str(dataset.get("description", ""))
    license_value   = str(dataset.get("license", "TBD"))
    keywords        = dataset.get("keywords", [])
    buckets         = dataset.get("buckets", {})
    dataset_doi     = dataset.get("doi", "")

    if not isinstance(keywords, list): keywords = [keywords]
    keywords = [str(k) for k in keywords if k is not None]
    if not isinstance(buckets, dict): buckets = {}

    tags              = get_tags(dataset)
    tags_search       = "||".join(t.lower() for t in tags)
    tags_html         = " ".join(f'<span class="tag-pill">{esc(t)}</span>' for t in tags) if tags else "NA"

    collection_name   = get_collection_name(dataset)
    dataset_version   = get_dataset_version(dataset)
    release_version   = get_release_version(dataset)
    collection_version = get_collection_version(dataset)
    collection_doi    = get_collection_doi(dataset)

    curation_release_history = get_curation_release_history(dataset)
    dataset_release_history  = get_dataset_release_history(dataset)
    all_releases             = get_all_releases(dataset)
    all_versions             = get_all_versions(dataset)

    # CDE version for the current/latest curation release
    curation          = get_curation(dataset)
    cur_rel_key       = curation.get("release_version", "")
    current_cde       = dataset_release_history.get(cur_rel_key, {}).get("cde_version", "")
    # All CDE versions across all releases for this dataset (for filter matching)
    all_cde_for_ds    = " ".join(
        ri.get("cde_version", "")
        for ri in dataset_release_history.values()
        if isinstance(ri, dict)
    )

    detail_id        = f"dataset-detail-{safe_id(dataset_id)}-{index}"
    panel_id_prefix  = f"ds-panel-{safe_id(dataset_id)}-{index}"

    # Search index
    curation_rel_search = " ".join(
        f"{rk} {ri.get('collection_version','')} {ri.get('release_version','')} {ri.get('collection_version_doi','')}"
        for rk, ri in curation_release_history.items() if isinstance(ri, dict)
    )
    dataset_rel_search = " ".join(
        f"{rk} {ri.get('dataset_version','')} {ri.get('cde_version','')}"
        for rk, ri in dataset_release_history.items() if isinstance(ri, dict)
    )
    search_text = " ".join([
        dataset_id, dataset_title, description, collection_name,
        dataset_version, release_version, collection_version, collection_doi, dataset_doi,
        " ".join(tags), " ".join(keywords),
        " ".join(all_releases), " ".join(all_versions),
        " ".join(str(v) for v in buckets.values()),
        curation_rel_search, dataset_rel_search,
    ]).lower()

    # Releases newest→oldest
    releases_sorted = sorted(dataset_release_history.keys(), key=version_key, reverse=True)

    # data-* attributes used by JS filters
    # data-release  = current/latest release version
    # data-cde      = pipe-separated list of ALL cde versions for this dataset
    # data-collection = collection name
    all_cde_attr = "||".join(
        ri.get("cde_version", "")
        for ri in dataset_release_history.values()
        if isinstance(ri, dict) and ri.get("cde_version")
    )

    # ── Summary row ─────────────────────────────────────────────────────────
    lines.extend([
        f'    <tr class="dataset-row"'
        f' data-detail="{esc(detail_id)}"'
        f' data-search="{esc(search_text)}"'
        f' data-tags="{esc(tags_search)}"'
        f' data-release="{esc(release_version)}"'
        f' data-cde="{esc(all_cde_attr)}"'
        f' data-collection="{esc(collection_name)}">',
        f"      <td><code>{esc(dataset_id)}</code></td>",
        f"      <td>{esc(dataset_title)}</td>",
        f"      <td>{esc(collection_name)}</td>",
        f"      <td>{esc(dataset_version)}</td>",
        f"      <td>{esc(release_version)}</td>",
        f"      <td>{esc(collection_version)}</td>",
        f'      <td><button class="dataset-toggle" data-target="{esc(detail_id)}">View</button></td>',
        "    </tr>",
    ])

    # ── Detail row ──────────────────────────────────────────────────────────
    meta_strip        = render_meta_strip(dataset_doi, license_value, collection_name, collection_doi, dataset_version)
    release_tabs      = render_release_tabs(releases_sorted, dataset, panel_id_prefix)
    release_panels    = render_release_panels(releases_sorted, dataset, buckets, panel_id_prefix)
    history_table     = render_full_history_table(releases_sorted, dataset, dataset_release_history, curation_release_history)
    collection_history = render_collection_release_history(curation_release_history, collection_name)

    lines.extend([
        f'    <tr id="{esc(detail_id)}" class="dataset-detail-row">',
        '      <td colspan="7">',
        '        <div class="dataset-detail">',
        f'          <h3 class="ds-title">{esc(dataset_title)}</h3>',
        f'          <div class="ds-id">{esc(dataset_id)}</div>',
        f'          <div class="ds-tag-row">{tags_html}</div>',
        meta_strip,
        '          <div class="ds-section">',
        '            <div class="ds-section-header">Description</div>',
        f'            <p class="ds-description">{esc(description) if description else "TBD"}</p>',
        '          </div>',
        '          <div class="ds-section">',
        '            <div class="ds-section-header">By release</div>',
        f'            {release_tabs}',
        f'            <div class="ds-panels" data-prefix="{esc(panel_id_prefix)}">',
        f'              {release_panels}',
        '            </div>',
        '          </div>',
        history_table,
        collection_history,
        '          <div class="ds-section">',
        '            <div class="ds-section-header">Source</div>',
        f'            <div class="ds-field"><span class="ds-field-label">Detail JSON</span>'
        f'<span class="ds-field-value ds-mono" style="font-size:0.7rem">{esc(dataset.get("_detail_file",""))}</span></div>',
        '          </div>',
        '        </div>',
        '      </td>',
        '    </tr>',
    ])

lines.extend(["  </tbody>", "</table>", ""])

OUT_FILE.parent.mkdir(parents=True, exist_ok=True)
OUT_FILE.write_text("\n".join(lines), encoding="utf-8")


# ----------------------------
# Generate JavaScript
# ----------------------------

js_text = r"""
// ─── Initialise dataset page ────────────────────────────────────────────────

function initializeDatasetPage() {
  var table = document.getElementById("datasetTable");
  if (!table) return;
  if (table.getAttribute("data-initialized") === "true") return;
  table.setAttribute("data-initialized", "true");

  var searchInput      = document.getElementById("datasetSearch");
  var tagFilter        = document.getElementById("tagFilter");
  var releaseFilter    = document.getElementById("releaseFilter");
  var cdeFilter        = document.getElementById("cdeFilter");
  var collectionFilter = document.getElementById("collectionFilter");
  var datasetCount     = document.getElementById("datasetCount");
  var rows             = Array.from(document.querySelectorAll(".dataset-row"));
  var buttons          = Array.from(document.querySelectorAll(".dataset-toggle"));

  if (!rows.length) return;

  // ── Count display ──────────────────────────────────────────────────────────
  function updateCount(n) {
    if (datasetCount) datasetCount.textContent = n + " of " + rows.length + " datasets shown";
  }

  // ── Close detail row ───────────────────────────────────────────────────────
  function closeDetail(row) {
    var id  = row.getAttribute("data-detail");
    var dr  = id ? document.getElementById(id) : null;
    var btn = row.querySelector(".dataset-toggle");
    if (dr)  dr.style.display = "none";
    if (btn) btn.textContent  = "View";
  }

  // ── Apply all active filters ───────────────────────────────────────────────
  function applyFilters() {
    var query      = searchInput      ? searchInput.value.toLowerCase().trim()      : "";
    var selTag     = tagFilter        ? tagFilter.value.toLowerCase().trim()        : "";
    var selRelease = releaseFilter    ? releaseFilter.value.toLowerCase().trim()    : "";
    var selCde     = cdeFilter        ? cdeFilter.value.toLowerCase().trim()        : "";
    var selCol     = collectionFilter ? collectionFilter.value.toLowerCase().trim() : "";

    var visible = 0;
    rows.forEach(function(row) {
      var text       = (row.getAttribute("data-search")     || "").toLowerCase();
      var tags       = (row.getAttribute("data-tags")       || "").toLowerCase();
      var rowRelease = (row.getAttribute("data-release")    || "").toLowerCase();
      var rowCde     = (row.getAttribute("data-cde")        || "").toLowerCase();
      var rowCol     = (row.getAttribute("data-collection") || "").toLowerCase();

      var tagList = tags.split("||").map(function(t){ return t.trim(); }).filter(Boolean);
      var cdeList = rowCde.split("||").map(function(t){ return t.trim(); }).filter(Boolean);

      var ok = (
        (query      === "" || text.includes(query))                       &&
        (selTag     === "" || tagList.includes(selTag))                   &&
        (selRelease === "" || rowRelease === selRelease)                   &&
        (selCde     === "" || cdeList.includes(selCde))                   &&
        (selCol     === "" || rowCol === selCol)
      );

      row.style.display = ok ? "table-row" : "none";
      if (!ok) closeDetail(row);
      if (ok)  visible++;
    });
    updateCount(visible);
  }

  // ── Toggle detail row ──────────────────────────────────────────────────────
  buttons.forEach(function(btn) {
    btn.addEventListener("click", function() {
      var id = btn.getAttribute("data-target");
      var dr = id ? document.getElementById(id) : null;
      if (!dr) return;
      var open = dr.style.display === "table-row";
      dr.style.display = open ? "none" : "table-row";
      btn.textContent  = open ? "View" : "Hide";
    });
  });

  // ── Wire up filters ────────────────────────────────────────────────────────
  [searchInput, tagFilter, releaseFilter, cdeFilter, collectionFilter].forEach(function(el) {
    if (el) el.addEventListener(el.tagName === "INPUT" ? "input" : "change", applyFilters);
  });

  // ── Sortable columns ───────────────────────────────────────────────────────
  var sortState = { col: -1, dir: "asc" };

  function versionVal(str) {
    // Parse "v4.0.2" → numeric tuple for comparison; fallback to string
    var nums = String(str).match(/\d+/g);
    if (!nums) return [0];
    return nums.map(Number);
  }

  function compareVersions(a, b) {
    var av = versionVal(a), bv = versionVal(b);
    for (var i = 0; i < Math.max(av.length, bv.length); i++) {
      var diff = (av[i] || 0) - (bv[i] || 0);
      if (diff !== 0) return diff;
    }
    return 0;
  }

  function sortTable(colIndex, dir) {
    var tbody = table.querySelector("tbody");
    // Collect pairs of [data-row, detail-row] to keep them together
    var pairs = [];
    var allRows = Array.from(tbody.querySelectorAll("tr"));
    for (var i = 0; i < allRows.length; i++) {
      if (allRows[i].classList.contains("dataset-row")) {
        var next = allRows[i + 1];
        pairs.push({
          main:   allRows[i],
          detail: (next && next.classList.contains("dataset-detail-row")) ? next : null
        });
      }
    }

    // Version-aware columns: 3 (dataset version), 4 (release), 5 (collection version)
    var versionCols = { 3: true, 4: true, 5: true };

    pairs.sort(function(a, b) {
      var aCell = a.main.querySelectorAll("td")[colIndex];
      var bCell = b.main.querySelectorAll("td")[colIndex];
      var aText = aCell ? aCell.textContent.trim() : "";
      var bText = bCell ? bCell.textContent.trim() : "";

      var cmp = versionCols[colIndex]
        ? compareVersions(aText, bText)
        : aText.toLowerCase().localeCompare(bText.toLowerCase());

      return dir === "asc" ? cmp : -cmp;
    });

    pairs.forEach(function(p) {
      tbody.appendChild(p.main);
      if (p.detail) tbody.appendChild(p.detail);
    });
  }

  table.querySelectorAll("th.sortable").forEach(function(th) {
    th.addEventListener("click", function() {
      var col = parseInt(th.getAttribute("data-col"), 10);
      var dir = (sortState.col === col && sortState.dir === "asc") ? "desc" : "asc";
      sortState = { col: col, dir: dir };

      // Reset all icons
      table.querySelectorAll(".sort-icon").forEach(function(ic) {
        ic.className = "sort-icon";
        ic.textContent = "⇅";
      });
      // Set active icon
      var icon = th.querySelector(".sort-icon");
      if (icon) {
        icon.className = "sort-icon " + dir;
        icon.textContent = dir === "asc" ? "↑" : "↓";
      }

      sortTable(col, dir);
    });
  });

  updateCount(rows.length);
}

// ─── Release tab switching ───────────────────────────────────────────────────
function dsTabSwitch(btn) {
  var panelId = btn.getAttribute("data-panel");
  if (!panelId) return;

  var tabGroup = btn.closest(".ds-release-tabs");
  if (tabGroup) {
    tabGroup.querySelectorAll(".ds-rtab").forEach(function(t) {
      t.classList.remove("ds-rtab--active");
    });
  }
  btn.classList.add("ds-rtab--active");

  var section = btn.closest(".ds-section");
  if (section) {
    section.querySelectorAll(".ds-panel-sections").forEach(function(p) {
      p.style.display = "none";
    });
  }
  var panel = document.getElementById(panelId);
  if (panel) panel.style.display = "flex";

  // Sync history table highlight — panel id ends with "-<release>"
  var parts   = panelId.split("-");
  var release = parts[parts.length - 1];
  var detailRow = btn.closest(".dataset-detail-row");
  if (detailRow) {
    detailRow.querySelectorAll(".ds-history-row").forEach(function(r) {
      r.classList.toggle("ds-history-row--active", r.getAttribute("data-release") === release);
    });
  }
}

// ─── Copy bucket path ────────────────────────────────────────────────────────
function dsCopyPath(btn) {
  var path = btn.getAttribute("data-path");
  if (!path) return;
  navigator.clipboard.writeText(path).catch(function(){});
  btn.textContent = "Copied!";
  btn.classList.add("ds-copy-btn--copied");
  setTimeout(function() {
    btn.textContent = "Copy";
    btn.classList.remove("ds-copy-btn--copied");
  }, 1500);
}

// ─── History row click → switch tab ─────────────────────────────────────────
document.addEventListener("click", function(e) {
  var row = e.target.closest(".ds-history-row");
  if (!row) return;
  var release = row.getAttribute("data-release");
  if (!release) return;
  var wrapper = row.closest(".dataset-detail");
  if (!wrapper) return;
  var tab = Array.from(wrapper.querySelectorAll(".ds-rtab")).find(function(t) {
    return (t.getAttribute("data-panel") || "").endsWith("-" + release);
  });
  if (tab) dsTabSwitch(tab);
});

// ─── MkDocs Material SPA compatibility ───────────────────────────────────────
if (typeof document$ !== "undefined") {
  document$.subscribe(function() { initializeDatasetPage(); });
} else {
  document.addEventListener("DOMContentLoaded", initializeDatasetPage);
}
"""

JS_FILE.parent.mkdir(parents=True, exist_ok=True)
JS_FILE.write_text(js_text.strip() + "\n", encoding="utf-8")

print(f"Wrote: {OUT_FILE}")
print(f"Wrote: {JS_FILE}")

Learning Lab root: /Users/amaraalexander/Documents/GitHub/asap-crn-learning-lab
Dataset repo: /Users/amaraalexander/Documents/GitHub/cloud-datasets
Collections repo: /Users/amaraalexander/Documents/GitHub/cloud-collections
Dataset index file: /Users/amaraalexander/Documents/GitHub/cloud-datasets/datasets.json
Dataset detail folder: /Users/amaraalexander/Documents/GitHub/cloud-datasets/datasets
Collections file: /Users/amaraalexander/Documents/GitHub/cloud-collections/collections.json
Output Markdown: /Users/amaraalexander/Documents/GitHub/asap-crn-learning-lab/docs/rosetta-stone/datasets.md
Output JavaScript: /Users/amaraalexander/Documents/GitHub/asap-crn-learning-lab/docs/javascripts/dataset-filter.js
Loaded 5 collections
Loaded 61 unique datasets
Loaded 48 unique tags
Releases for filter: ['v4.1.1', 'v4.1.0', 'v4.0.2', 'v4.0.1', 'v4.0.0']
CDE versions for filter: ['v4.4', 'v4.3', 'v4.2', 'v4.1', 'v3.3', 'v3.2', 'v3.1', 'v3.0', 'v2.1']
Collections for filter: ['mouse-sc-rnaseq', 'mou